In [25]:
import os
import numpy as np
import pandas as pd
from absl import app
from absl import flags
from typing import Sequence
from tqdm import tqdm
import matplotlib.pyplot as plt
import parameterized_sample_lib as psample
import cat_machine_contest_metrics as cmcm

In [ ]:
params_list = [] 
all_params_list = [] 
ks = [x for x in range(1,11)] 
ks.extend([x for x in range(20,1001, 20)])

def get_n_k_for_num_ratings(all_values, num_ratings=5000): 
    values=[] 
    for x in all_values:
        if x==0:
            x=1 
        n = int(np.floor(num_ratings/x)) 
        # if n > 30:
        values.append((n, int(x))) 
    return values
# vals = get_n_k_for_num_ratings(ks) 
# [x[1] for x in vals], [x[0] for x in vals]
ratings_list = [] 
# nk_list = [5000]
nk_list = [2500, 5000, 10000, 25000, 50000]
# nk_list = [1000, 2500, 5000, 10000, 25000, 50000]
# for r in range(1000, 5001, 1000):
# for r in range(5000, 5001, 500): 
for r in nk_list:
    params = get_n_k_for_num_ratings(ks, r) 
    params_list.append(params)
    all_params_list.extend(params) 
    ratings_list.extend([(r-x[0]*x[1]) for x in params])
    print(params)

[(5000, 1), (2500, 2), (1666, 3), (1250, 4), (1000, 5), (833, 6), (714, 7), (625, 8), (555, 9), (500, 10), (250, 20), (125, 40), (83, 60), (62, 80), (50, 100), (41, 120), (35, 140), (31, 160), (27, 180), (25, 200), (22, 220), (20, 240), (19, 260), (17, 280), (16, 300), (15, 320), (14, 340), (13, 360), (13, 380), (12, 400), (11, 420), (11, 440), (10, 460), (10, 480), (10, 500), (9, 520), (9, 540), (8, 560), (8, 580), (8, 600), (8, 620), (7, 640), (7, 660), (7, 680), (7, 700), (6, 720), (6, 740), (6, 760), (6, 780), (6, 800), (6, 820), (5, 840), (5, 860), (5, 880), (5, 900), (5, 920), (5, 940), (5, 960), (5, 980), (5, 1000)]


In [ ]:
def gather_data(_N_ITEMS, _K_RESPONSES, distortion_values, exp_dir, metrics_list, _M_CATEGORIES=3):
    final_table = pd.DataFrame()
    for distortion in distortion_values:
        experiment_results = pd.read_csv(f'{exp_dir}results_N={_N_ITEMS}_K={_K_RESPONSES}_cat_responses_simulated_distr_dist={distortion}_gen_N={_N_ITEMS}_K={_K_RESPONSES}_M={_M_CATEGORIES}_num_samples=1000.pkl.csv')
        intermediate_table = pd.DataFrame()
        intermediate_table['$\\Delta$'] = (experiment_results['M2 GT Alt'] - experiment_results['M1 GT Alt']).abs()
        intermediate_table['p-value'] = experiment_results['GT_Pvalue']
        # intermediate_table['Metric'] = ['$\\Gamma_{\\rm Accuracy}$', '$\\Gamma_{\\rm F1-score}$']
        intermediate_table['Metric'] = metrics_list
        intermediate_table[f'$\\epsilon$'] = distortion
        final_table = pd.concat([final_table, intermediate_table])

    final_table = final_table.melt(["Metric", "$\\epsilon$"]).sort_values(by=["Metric","variable"]).pivot(index = "$\\epsilon$", columns=["Metric","variable"])
    # final_table = final_table.reset_index(drop=True)
    final_table = final_table.reset_index()
    final_table.columns = pd.MultiIndex.from_tuples([(j,k) for i,j,k in final_table.columns])
    final_table.columns = ['_'.join(col) for col in final_table.columns]
    final_table["N"] = pd.Series([_N_ITEMS]*len(distortion_values))
    final_table["K"] = pd.Series([_K_RESPONSES]*len(distortion_values))
    final_table["NxK"] = final_table["N"]*final_table["K"]

    return final_table

In [28]:
def get_metric_scores(response_sets, metric):
    scores = []
    for i in range(1000):
        score=(0.0,0.0)

        simulated_data_gold = response_sets.alt_data_list[i].gold
        simulated_data_preds1 = response_sets.alt_data_list[i].preds1
        simulated_data_preds2 = response_sets.alt_data_list[i].preds2

        if metric=="Accuracy":
            score = cmcm.cat_accuracy(simulated_data_gold, simulated_data_preds1, simulated_data_preds2)
        if metric=="MAE":
            score = cmcm.cat_mean_absolute_error(simulated_data_gold, simulated_data_preds1, simulated_data_preds2)
        if metric=="Wins":
            score = cmcm.cat_wins_mae(simulated_data_gold, simulated_data_preds1, simulated_data_preds2)
        if metric=="KL-Div":
            score = cmcm.cat_kl_div(simulated_data_gold, simulated_data_preds1, simulated_data_preds2)
        scores.append(score)
    return scores

In [29]:
def plot_histogram(n_items, k_responses, scores, metric, distortion, dataset, final_table, base_path):
    scores_m1 = [x[0] for x in scores]
    scores_m2 = [x[1] for x in scores]

    mean_m1 = np.mean(scores_m1)
    var_m1 = np.var(scores_m1)

    mean_m2 = np.mean(scores_m2)
    var_m2 = np.var(scores_m2)

    plt.figure(figsize=(10, 6))
    plt.hist(scores_m1, bins='auto', alpha=0.5, label=f'Machine 1 (μ={mean_m1:.4f}, σ²={var_m1:.6f})')
    plt.hist(scores_m2, bins='auto', alpha=0.5, label=f'Machine 2 (μ={mean_m2:.4f}, σ²={var_m2:.6f})')
    plt.xlabel('Score')
    plt.ylabel('Freq')
    if metric=='Accuracy':
        plt.xlim((0.4,1))
    if metric=='MAE':
        plt.xlim((0,0.5))

    plt.legend(loc='upper right')
    plt.title(f"{dataset} - Distribution of {metric} ($\\epsilon$={distortion}, n={n_items}, k={k_responses})")

    # plt.text(0.02, plt.ylim()[1]*0.9, f'M1: μ={mean_m1:.2f}, σ²={var_m1:.6f}', fontsize=9, color='blue')
    # plt.text(0.02, plt.ylim()[1]*0.8, f'M2: μ={mean_m2:.2f}, σ²={var_m2:.6f}', fontsize=9, color='orange')
    # plt.text((mean_m1+mean_m2)/2+0.05, plt.ylim()[1]*0.8, f'p-val: {float(final_table[f"{metric}_p-value"]):.6f}', fontsize=9, color='green')
    plt.text(plt.xlim()[1]*0.8, plt.ylim()[1]*0.8, f'p-val: {float(final_table[f"{metric}_p-value"].iloc[0]):.6f}', fontsize=9, color='green')

    # plt.tight_layout()
    plt.savefig(f"{base_path}/{dataset}_dist_n={n_items}_k={k_responses}_{metric}_e_{distortion}.png")
    plt.close()

In [ ]:
dataset_info = {"exp_dir": "../../../../data/ptest_arr_dices/", "dataset": "DICES", "num_categories": "3",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_d3code/", "dataset": "D3code", "num_categories": "2",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_jobsQ1/", "dataset": "JobsQ1", "num_categories": "5",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_jobsQ3/", "dataset": "JobsQ3", "num_categories": "12",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_toxicity/", "dataset": "toxicity", "num_categories": "2",}

# dataset_info = {"exp_dir": "../../../../data/ptest_arr_uniform/", "dataset": "uniform", "num_categories": "2",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_gamma/", "dataset": "gamma", "num_categories": "3",}

# dataset_info = {"exp_dir": "../../../../data/ptest_arr_dices_actual_p/", "dataset": "DICES actual p-vals", "num_categories": "3",}


# metrics_list = ['Accuracy']
metrics_list = ['MAE']
# metrics_list = ['Accuracy', 'MAE']
# metrics_list = ['Accuracy', 'MAE', 'Wins', 'KL-Div']

n_items = 5000
k_responses = 1
distortion=0.1
# distortion_values = [0.1]

base_path = "output/hist_plots"
if not os.path.exists(base_path):
    os.mkdir(base_path)


for n_items, k_responses in tqdm(all_params_list[:30], desc="Generating plots"):
    dataset=dataset_info['dataset']
    m_categories = dataset_info['num_categories']
    exp_dir = dataset_info['exp_dir']

    response_sets = psample.read_samples_from_file(f"{exp_dir}cat_responses_simulated_distr_dist={distortion}_gen_N={n_items}_K={k_responses}_M={m_categories}_num_samples=1000.pkl", True)
    response_sets.truncate(n_items, k_responses)

    final_table = gather_data(n_items, k_responses, [distortion], exp_dir, metrics_list)

    for metric in metrics_list:
        scores = get_metric_scores(response_sets, metric)
        plot_histogram(n_items, k_responses, scores, metric, distortion, dataset, final_table, base_path)
        # break
    
    # break


100%|██████████| 30/30 [02:33<00:00,  5.10s/it]
